In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
folder = "/content/drive/MyDrive/MS Spring 2026 (Y1 Q3)/ADSP 31012 DEPA Final Project"

In [23]:
import pandas as pd
import numpy as np
import re

In [6]:
# Load the clean CSV saved from EDA
#BRFSS = pd.read_csv(f"{folder}/EDA and Cleanup/BRFSS_2024_clean.csv")

df_raw = pd.read_sas(folder + "/Raw Data/LLCP2024.XPT_", format="xport", encoding="latin1")
print(df_raw.shape)
print(df_raw.head())

(457670, 301)
   _STATE  FMONTH     IDATE IMONTH IDAY IYEAR  DISPCODE       SEQNO  \
0     1.0     2.0  02282024     02   28  2024    1100.0  2024000001   
1     1.0     2.0  02212024     02   21  2024    1100.0  2024000002   
2     1.0     2.0  02212024     02   21  2024    1100.0  2024000003   
3     1.0     2.0  02282024     02   28  2024    1100.0  2024000004   
4     1.0     2.0  02212024     02   21  2024    1100.0  2024000005   

           _PSU  CTELENM1  ...  _LCSCTSN  _LCSPSTF  DRNKANY6      DROCDY4_  \
0  2.024000e+09       1.0  ...       NaN       9.0       2.0  5.397605e-79   
1  2.024000e+09       1.0  ...       4.0       9.0       2.0  5.397605e-79   
2  2.024000e+09       1.0  ...       4.0       2.0       1.0  1.000000e+02   
3  2.024000e+09       1.0  ...       NaN       9.0       2.0  5.397605e-79   
4  2.024000e+09       1.0  ...       3.0       9.0       2.0  5.397605e-79   

   _RFBING6      _DRNKWK3  _RFDRHV9  _FLSHOT7  _PNEUMO3  _AIDTST4  
0       1.0  5.397605e

In [7]:
NEEDED = [
    'SEQNO','_STATE','_LLCPWT','IMONTH','IDAY','IYEAR',
    '_METSTAT','_URBSTAT','MSCODE',
    '_SEX','_AGEG5YR','_RACE','_HISPANC','_EDUCAG','_INCOMG1',
    'MARITAL','EMPLOY1','CHILDREN','VETERAN3','PREGNANT',
    'GENHLTH','PHYSHLTH','MENTHLTH','_BMI5','_BMI5CAT',
    'CVDINFR4','CVDSTRK3','DIABETE4','ASTHMA3','ADDEPEV3',
    'CHCCOPD3','HAVARTH4','CHCOCNC1','CHCKDNY2','LSATISFY',
    'EXERANY2','_SMOKER3','ECIGNOW3','DRNKANY6','_RFBING6',
    '_RFDRHV9','AVEDRNK4','MARIJAN1','SSBSUGR2','FIREARM5',
    '_HLTHPL2','PRIMINS2','PERSDOC3','MEDCOST1','CHECKUP1',
    'LASTDEN4','_RFMAM23','_CRVSCRN','_CRCREC3',
    '_FLSHOT7','_PNEUMO3','_AIDTST4','SDHFOOD1','SDHTRNSP',
    'QSTVER','QSTLANG','CELLFON5'
]

df = df_raw[NEEDED].copy()

print(f"Raw shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Raw shape: (457670, 62)
Columns: ['SEQNO', '_STATE', '_LLCPWT', 'IMONTH', 'IDAY', 'IYEAR', '_METSTAT', '_URBSTAT', 'MSCODE', '_SEX', '_AGEG5YR', '_RACE', '_HISPANC', '_EDUCAG', '_INCOMG1', 'MARITAL', 'EMPLOY1', 'CHILDREN', 'VETERAN3', 'PREGNANT', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', '_BMI5', '_BMI5CAT', 'CVDINFR4', 'CVDSTRK3', 'DIABETE4', 'ASTHMA3', 'ADDEPEV3', 'CHCCOPD3', 'HAVARTH4', 'CHCOCNC1', 'CHCKDNY2', 'LSATISFY', 'EXERANY2', '_SMOKER3', 'ECIGNOW3', 'DRNKANY6', '_RFBING6', '_RFDRHV9', 'AVEDRNK4', 'MARIJAN1', 'SSBSUGR2', 'FIREARM5', '_HLTHPL2', 'PRIMINS2', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1', 'LASTDEN4', '_RFMAM23', '_CRVSCRN', '_CRCREC3', '_FLSHOT7', '_PNEUMO3', '_AIDTST4', 'SDHFOOD1', 'SDHTRNSP', 'QSTVER', 'QSTLANG', 'CELLFON5']


This script transforms the raw BRFSS 2024 XPT file into a clean, analysis-ready DataFrame by selecting the 63 needed columns, recoding all numeric survey codes to readable labels and proper nulls, fixing a broken diabetes recode, and creating 8 derived columns for richer analysis. Every change is validated at the end by confirming row counts are preserved and all 6 weighted disease prevalences match the known EDA benchmarks exactly.

In [8]:
# ── STEP 1: RECODE SKIP CODES TO NULL ────────────────────────────────────────
SKIP_CODES = {
    'GENHLTH':   [7, 9],
    'PHYSHLTH':  [77, 99],
    'MENTHLTH':  [77, 99],
    'LSATISFY':  [7, 9],
    'CVDINFR4':  [7, 9],
    'CVDSTRK3':  [7, 9],
    'DIABETE4':  [7, 9],
    'ASTHMA3':   [7, 9],
    'ADDEPEV3':  [7, 9],
    'CHCCOPD3':  [7, 9],
    'HAVARTH4':  [7, 9],
    'CHCOCNC1':  [7, 9],
    'CHCKDNY2':  [7, 9],
    'EXERANY2':  [7, 9],
    '_SMOKER3':  [9],
    'ECIGNOW3':  [7, 9],
    'DRNKANY6':  [7, 9],
    '_RFBING6':  [9],
    '_RFDRHV9':  [9],
    'AVEDRNK4':  [77, 99],
    'MARIJAN1':  [7, 9],
    'SSBSUGR2':  [7777, 9999],
    'FIREARM5':  [7, 9],
    '_HLTHPL2':  [9],
    'PRIMINS2':  [77, 99],
    'PERSDOC3':  [7, 9],
    'MEDCOST1':  [7, 9],
    'CHECKUP1':  [7, 9],
    'LASTDEN4':  [7, 9],
    '_RFMAM23':  [9],
    '_CRVSCRN':  [9],
    '_CRCREC3':  [9],
    '_FLSHOT7':  [9],
    '_PNEUMO3':  [9],
    '_AIDTST4':  [9],
    'SDHFOOD1':  [7, 9],
    'SDHTRNSP':  [7, 9],
    'MARITAL':   [9],
    'EMPLOY1':   [9],
    'CHILDREN':  [99],
    'VETERAN3':  [7, 9],
    'PREGNANT':  [7, 9],
    '_SEX':      [9],
    '_AGEG5YR':  [14],
    '_RACE':     [9],
    '_HISPANC':  [7, 9],
    '_EDUCAG':   [9],
    '_INCOMG1':  [9],
    '_BMI5CAT':  [9],
    'CELLFON5':  [7, 9],
    'MSCODE':    [9],
    '_METSTAT':  [9],
    '_URBSTAT':  [9],
}

for col, codes in SKIP_CODES.items():
    if col in df.columns:
        df[col] = df[col].replace(codes, np.nan)

print("Step 1 complete: skip codes recoded to NaN")

# ── STEP 2: RECODE BINARY YES/NO TO TRUE/FALSE ───────────────────────────────
BINARY_COLS = [
    'CVDINFR4', 'CVDSTRK3', 'ASTHMA3', 'ADDEPEV3',
    'CHCCOPD3', 'HAVARTH4', 'CHCOCNC1', 'CHCKDNY2',
    'EXERANY2', 'DRNKANY6', 'MEDCOST1', 'VETERAN3', 'PREGNANT'
]
for col in BINARY_COLS:
    if col in df.columns:
        df[col] = df[col].map({1.0: True, 2.0: False})

print("Step 2 complete: binary columns recoded")

# ── STEP 3: RECODE DIABETE4 ───────────────────────────────────────────────────
df['diabetes_status'] = df['DIABETE4'].map({
    1.0: 'Diabetic',
    2.0: 'Gestational Only',
    3.0: 'No',
    4.0: 'Pre-Diabetic',
})

df['diabetes'] = df['DIABETE4'].map({
    1.0: True,
    2.0: False,  # Gestational Only → not currently diabetic
    3.0: False,
    4.0: False,
})

df = df.drop(columns=['DIABETE4'])
print("Step 3 complete: DIABETE4 recoded, dropped")

# ── STEP 4: RECODE CATEGORICALS TO LABELS ────────────────────────────────────
df['_SEX'] = df['_SEX'].map({1.0: 'Male', 2.0: 'Female'})

df['_AGEG5YR'] = df['_AGEG5YR'].map({
    1.0: '18-24', 2.0: '25-29', 3.0: '30-34', 4.0: '35-39',
    5.0: '40-44', 6.0: '45-49', 7.0: '50-54', 8.0: '55-59',
    9.0: '60-64', 10.0: '65-69', 11.0: '70-74', 12.0: '75-79',
    13.0: '80+'
})

df['_RACE'] = df['_RACE'].map({
    1.0: 'White NH', 2.0: 'Black NH', 3.0: 'AIAN NH',
    4.0: 'Asian NH', 5.0: 'NHOPI NH', 6.0: 'Other NH',
    7.0: 'Multiracial NH', 8.0: 'Hispanic'
})

df['_HISPANC'] = df['_HISPANC'].map({1.0: 'Hispanic', 2.0: 'Not Hispanic'})

df['_EDUCAG'] = df['_EDUCAG'].map({
    1.0: '< HS', 2.0: 'HS Grad',
    3.0: 'Some College', 4.0: 'College Grad'
})

df['_INCOMG1'] = df['_INCOMG1'].map({
    1.0: '<$15k', 2.0: '$15-25k', 3.0: '$25-35k', 4.0: '$35-50k',
    5.0: '$50-100k', 6.0: '$100-200k', 7.0: '>$200k'
})

df['MARITAL'] = df['MARITAL'].map({
    1.0: 'Married', 2.0: 'Divorced', 3.0: 'Widowed',
    4.0: 'Separated', 5.0: 'Never Married', 6.0: 'Unmarried Couple'
})

df['EMPLOY1'] = df['EMPLOY1'].map({
    1.0: 'Employed', 2.0: 'Self-Employed', 3.0: 'Unemployed <1yr',
    4.0: 'Unemployed >1yr', 5.0: 'Homemaker', 6.0: 'Student',
    7.0: 'Retired', 8.0: 'Unable to Work'
})

df['GENHLTH'] = df['GENHLTH'].map({
    1.0: 'Excellent', 2.0: 'Very Good', 3.0: 'Good',
    4.0: 'Fair', 5.0: 'Poor'
})

df['LSATISFY'] = df['LSATISFY'].map({
    1.0: 'Very Satisfied', 2.0: 'Satisfied',
    3.0: 'Dissatisfied', 4.0: 'Very Dissatisfied'
})

df['_BMI5CAT'] = df['_BMI5CAT'].map({
    1.0: 'Underweight', 2.0: 'Normal weight',
    3.0: 'Overweight', 4.0: 'Obese'
})

df['_SMOKER3'] = df['_SMOKER3'].map({
    1.0: 'Every day', 2.0: 'Some days',
    3.0: 'Former', 4.0: 'Never'
})

df['ECIGNOW3'] = df['ECIGNOW3'].map({
    1.0: 'Every Day', 2.0: 'Some Days', 3.0: 'Not at All'
})

df['_RFBING6'] = df['_RFBING6'].map({1.0: False, 2.0: True})
df['_RFDRHV9'] = df['_RFDRHV9'].map({1.0: False, 2.0: True})
df['_HLTHPL2'] = df['_HLTHPL2'].map({1.0: 'Insured', 2.0: 'Not Insured'})

df['PERSDOC3'] = df['PERSDOC3'].map({
    1.0: 'Yes - Only One', 2.0: 'Yes - More Than One', 3.0: 'No'
})

df['CHECKUP1'] = df['CHECKUP1'].map({
    1.0: 'Past Year', 2.0: 'Past 2 Years',
    3.0: 'Past 5 Years', 4.0: '5+ Years Ago', 8.0: 'Never'
})

df['LASTDEN4'] = df['LASTDEN4'].map({
    1.0: 'Past Year', 2.0: 'Past 2 Years',
    3.0: 'Past 5 Years', 4.0: '5+ Years Ago', 8.0: 'Never'
})

df['SDHFOOD1'] = df['SDHFOOD1'].map({
    1.0: 'Always', 2.0: 'Usually', 3.0: 'Sometimes',
    4.0: 'Rarely', 5.0: 'Never'
})

df['_METSTAT'] = df['_METSTAT'].map({1.0: 'Metro', 2.0: 'Non-Metro'})
df['_URBSTAT'] = df['_URBSTAT'].map({1.0: 'Urban', 2.0: 'Rural'})
df['SDHTRNSP'] = df['SDHTRNSP'].map({1.0: True, 2.0: False})
df['_RFMAM23'] = df['_RFMAM23'].map({1.0: False, 2.0: True})
df['_CRVSCRN'] = df['_CRVSCRN'].map({1.0: True, 2.0: False})
df['_CRCREC3'] = df['_CRCREC3'].map({1.0: True, 2.0: False})
df['_FLSHOT7'] = df['_FLSHOT7'].map({1.0: True, 2.0: False})
df['_PNEUMO3'] = df['_PNEUMO3'].map({1.0: True, 2.0: False})
df['_AIDTST4'] = df['_AIDTST4'].map({1.0: True, 2.0: False})
df['CELLFON5'] = df['CELLFON5'].map({1.0: True, 2.0: False})

print("Step 4 complete: categoricals recoded")

# ── STEP 5: DERIVED COLUMNS ───────────────────────────────────────────────────

# physhlth_bin & menthlth_bin
def bin_health_days(val):
    if pd.isna(val): return np.nan
    val = int(val)
    if val == 0:    return 'None'
    elif val <= 7:  return 'Low'
    elif val <= 14: return 'Moderate'
    elif val <= 29: return 'High'
    else:           return 'Chronic'

df['physhlth_bin'] = df['PHYSHLTH'].apply(bin_health_days)
df['menthlth_bin'] = df['MENTHLTH'].apply(bin_health_days)

# bmi_category_clean — _BMI5 is stored as BMI × 100 (e.g. 2663 = 26.63)
df['bmi_category_clean'] = df['_BMI5'].apply(
    lambda x: np.nan if pd.isna(x)
    else 'Underweight' if (x / 100) < 18.5
    else 'Normal weight' if (x / 100) < 25
    else 'Overweight' if (x / 100) < 30
    else 'Obese'
)
df['bmi_category_clean'] = df['bmi_category_clean'].fillna(df['_BMI5CAT'])

# ever_smoker
df['ever_smoker'] = df['_SMOKER3'].map({
    'Every day': True, 'Some days': True,
    'Former': True, 'Never': False
})

# log_weight
df['log_weight'] = np.log1p(df['_LLCPWT'])

# condition_count
CONDITION_COLS = [
    'CVDINFR4', 'CVDSTRK3', 'ASTHMA3', 'ADDEPEV3',
    'CHCCOPD3', 'HAVARTH4', 'CHCOCNC1', 'CHCKDNY2'
]
df['condition_count'] = (
    df[CONDITION_COLS].apply(lambda x: x.astype(float)).sum(axis=1, skipna=True)
    + df['diabetes'].astype(float).fillna(0)
)

# age_decade
AGE_DECADE_MAP = {
    '18-24': '18-34', '25-29': '18-34', '30-34': '18-34',
    '35-39': '35-54', '40-44': '35-54', '45-49': '35-54', '50-54': '35-54',
    '55-59': '55-74', '60-64': '55-74', '65-69': '55-74', '70-74': '55-74',
    '75-79': '75+',   '80+':   '75+'
}
df['age_decade'] = df['_AGEG5YR'].map(AGE_DECADE_MAP)

# quarter
df['IMONTH'] = pd.to_numeric(df['IMONTH'], errors='coerce')
df['quarter'] = df['IMONTH'].apply(
    lambda m: np.nan if pd.isna(m)
    else 'Q1' if m <= 3
    else 'Q2' if m <= 6
    else 'Q3' if m <= 9
    else 'Q4'
)

print("Step 5 complete: derived columns created")

# ── STEP 6: VALIDATE ──────────────────────────────────────────────────────────
print(f"\nFinal shape: {df.shape}")

print("\ndiabetes_status:")
print(df['diabetes_status'].value_counts(dropna=False))

print("\nbmi_category_clean:")
print(df['bmi_category_clean'].value_counts(dropna=False))

print("\nDerived column summary:")
derived_cols = [
    'physhlth_bin', 'menthlth_bin', 'bmi_category_clean',
    'ever_smoker', 'log_weight', 'condition_count',
    'age_decade', 'quarter'
]
for col in derived_cols:
    null_pct = df[col].isna().mean() * 100
    print(f"  {col:25s} — {df[col].nunique():3} unique, {null_pct:.1f}% null")

print("\n── Weighted Prevalence Spot Check ──")
checks = {
    'Arthritis':    'HAVARTH4',
    'Depression':   'ADDEPEV3',
    'Diabetes':     'diabetes',
    'Asthma':       'ASTHMA3',
    'Heart Attack': 'CVDINFR4',
    'Stroke':       'CVDSTRK3',
}
for name, col in checks.items():
    mask = df[col] == True
    prev = (df.loc[mask, '_LLCPWT'].sum() / df['_LLCPWT'].sum()) * 100
    print(f"  {name:15s}: {prev:.1f}%")

print("\nExpected:")
print("  Arthritis 26.4% | Depression 20.9% | Diabetes 12.5%")
print("  Asthma 15.7%    | Heart Attack 4.3% | Stroke 3.5%")

print("\ncondition_count distribution:")
print(df['condition_count'].value_counts().sort_index())

Step 1 complete: skip codes recoded to NaN
Step 2 complete: binary columns recoded
Step 3 complete: DIABETE4 recoded, dropped
Step 4 complete: categoricals recoded
Step 5 complete: derived columns created

Final shape: (457670, 71)

diabetes_status:
diabetes_status
No                  376125
Diabetic             65809
Pre-Diabetic         11307
Gestational Only      3395
NaN                   1034
Name: count, dtype: int64

bmi_category_clean:
bmi_category_clean
Overweight       146563
Obese            139640
Normal weight    121053
NaN               43037
Underweight        7377
Name: count, dtype: int64

Derived column summary:
  physhlth_bin              —   4 unique, 2.4% null
  menthlth_bin              —   4 unique, 1.8% null
  bmi_category_clean        —   4 unique, 9.4% null
  ever_smoker               —   2 unique, 7.0% null
  log_weight                — 256420 unique, 0.0% null
  condition_count           —  10 unique, 0.0% null
  age_decade                —   4 unique, 1.8% 

## Recoded Columns

| Column | What Was Done |
|---|---|
| Skip Codes | Replaced non-answer codes (7, 9, 77, 99) with `NaN` to prevent corrupt calculations |
| Binary Yes/No | Converted Yes=1/No=2 to True/False across all binary condition and behavior columns |
| `DIABETE4` | Re-mapped all 4 codes into `diabetes` (True/False) and `diabetes_status` (Diabetic/Pre-Diabetic/Gestational/No) to fix 388K false NaNs |
| `_SEX` | Mapped 1/2 to Male/Female |
| `_AGEG5YR` | Mapped 1–13 to readable age bands (18-24 through 80+) |
| `_RACE` / `_HISPANC` | Mapped numeric codes to race/ethnicity labels |
| `_EDUCAG` / `_INCOMG1` | Mapped to education level and income bracket labels |
| `MARITAL` / `EMPLOY1` | Mapped to marital status and employment category labels |
| `GENHLTH` / `LSATISFY` | Mapped 1–5 scales to Excellent/Very Good/Good/Fair/Poor and satisfaction labels |
| `_BMI5CAT` / `_SMOKER3` / `ECIGNOW3` | Mapped BMI category, smoking status, and e-cigarette use to readable labels |
| `CHECKUP1` / `LASTDEN4` | Mapped recency codes to Past Year/Past 2 Years/Past 5 Years/5+ Years Ago/Never |
| `PERSDOC3` / `_HLTHPL2` | Mapped personal doctor and insurance status to readable labels |
| `SDHFOOD1` / `_METSTAT` / `_URBSTAT` | Mapped food insecurity frequency and metro/urban status to labels |
| Preventive Care Flags | Mapped `_RFMAM23`, `_CRVSCRN`, `_CRCREC3`, `_FLSHOT7`, `_PNEUMO3`, `_AIDTST4` from 1/2 to True/False |

## Derived Columns

| Column | What Was Done | Reasoning |
|---|---|---|
| `physhlth_bin` / `menthlth_bin` | Binned 0–30 poor health days into None/Low/Moderate/High/Chronic | Raw scale is skewed toward 0 and 30 — bins create meaningful OLAP groupings for dashboard filtering |
| `bmi_category_clean` | Re-derived from raw `_BMI5` (÷100) using WHO cutoffs | `_BMI5CAT` had 9.4% missing — re-deriving recovers those records |
| `ever_smoker` | True if daily/some days/former, False if never | Avoids repeated CASE WHEN logic in every query |
| `log_weight` | log(1 + `_LLCPWT`) | Raw weight is heavily right-skewed — log transform normalizes it for visualization and modeling |
| `condition_count` | Sum of all 9 binary condition flags per respondent (0–9) | Multimorbidity score doesn't exist in raw data but is a standard epidemiological metric |
| `age_decade` | Collapsed 13 five-year bands into 4 groups (18–34, 35–54, 55–74, 75+) | Five-year bands are too granular for dashboards — enables summary-to-detail drill-down |
| `quarter` | Q1–Q4 derived from interview month | Enables time-based filtering even though BRFSS is an annual survey |

In [29]:
import re
import numpy as np

# ── TRANSFORM MEDICAID ────────────────────────────────────────

# Step 1: Keep only needed columns and rename
medicaid = medicaid[['Location', 'Status of Medicaid Expansion Decision',
                     'Expansion Implementation Date']].copy()
medicaid.columns = ['state_name', 'medicaid_expansion_status', 'expansion_year']

# Step 2: Drop United States summary row
medicaid = medicaid[medicaid['state_name'] != 'United States'].copy().reset_index(drop=True)

# Step 3: Extract year
def extract_year(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    match = re.search(r'\b(20\d{2})\b', val)
    if match:
        return int(match.group(1))
    match = re.search(r'\d+/\d+/(\d{2})$', val)
    if match:
        return 2000 + int(match.group(1))
    return np.nan

medicaid['expansion_year'] = medicaid['expansion_year'].apply(extract_year)

# Step 4: Add territories
territories = pd.DataFrame({
    'state_name': ['Puerto Rico', 'Guam', 'Virgin Islands'],
    'medicaid_expansion_status': ['Not Applicable', 'Not Applicable', 'Not Applicable'],
    'expansion_year': [np.nan, np.nan, np.nan]
})
medicaid = pd.concat([medicaid, territories], ignore_index=True)

# Validate
print(f"Shape: {medicaid.shape}")
print(f"\nmedicaid_expansion_status:")
print(medicaid['medicaid_expansion_status'].value_counts(dropna=False))
print(f"\nexpansion_year:")
print(medicaid['expansion_year'].value_counts(dropna=False))
print(f"\nSample:")
print(medicaid.head(10))

Shape: (54, 3)

medicaid_expansion_status:
medicaid_expansion_status
Adopted           41
Not Adopted       10
Not Applicable     3
Name: count, dtype: int64

expansion_year:
expansion_year
2014.0    27
NaN       13
2015.0     3
2020.0     3
2016.0     2
2019.0     2
2021.0     2
2023.0     2
Name: count, dtype: int64

Sample:
             state_name medicaid_expansion_status  expansion_year
0               Alabama               Not Adopted             NaN
1                Alaska                   Adopted          2015.0
2               Arizona                   Adopted          2014.0
3              Arkansas                   Adopted          2014.0
4            California                   Adopted          2014.0
5              Colorado                   Adopted          2014.0
6           Connecticut                   Adopted          2014.0
7              Delaware                   Adopted          2014.0
8  District of Columbia                   Adopted          2014.0
9          

## Chosen Columns

| Column | Renamed To | Reason |
|---|---|---|
| `Location` | `state_name` | Join key to BRFSS and SVI on state name |
| `Status of Medicaid Expansion Decision` | `medicaid_expansion_status` | Core policy flag — Adopted / Not Adopted / Not Applicable |
| `Expansion Implementation Date` | `expansion_year` | Year expansion took effect — states that expanded in 2014 vs 2023 have very different coverage histories |

## Derived

| Derivation | What Was Done | Reasoning |
|---|---|---|
| `expansion_year` | Extracted 4-digit year from date strings (e.g. `9/1/15` → `2015`) | Raw format was inconsistent strings — integer year is cleaner for filtering and joining |
| Territories added | Added Puerto Rico, Guam, Virgin Islands as `Not Applicable` | KFF only tracks 50 states + DC — territories exist in BRFSS and need a matching row to avoid orphan FK keys |
| United States row dropped | Removed summary row | Aggregate row — not a state, would corrupt any state-level join |

In [19]:
svi = pd.read_csv(folder + "/Raw Data/SVI_2022_US_county.csv")
print(svi.shape)
print(svi.columns.tolist())

(3144, 158)
['ST', 'STATE', 'ST_ABBR', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION', 'AREA_SQMI', 'E_TOTPOP', 'M_TOTPOP', 'E_HU', 'M_HU', 'E_HH', 'M_HH', 'E_POV150', 'M_POV150', 'E_UNEMP', 'M_UNEMP', 'E_HBURD', 'M_HBURD', 'E_NOHSDP', 'M_NOHSDP', 'E_UNINSUR', 'M_UNINSUR', 'E_AGE65', 'M_AGE65', 'E_AGE17', 'M_AGE17', 'E_DISABL', 'M_DISABL', 'E_SNGPNT', 'M_SNGPNT', 'E_LIMENG', 'M_LIMENG', 'E_MINRTY', 'M_MINRTY', 'E_MUNIT', 'M_MUNIT', 'E_MOBILE', 'M_MOBILE', 'E_CROWD', 'M_CROWD', 'E_NOVEH', 'M_NOVEH', 'E_GROUPQ', 'M_GROUPQ', 'EP_POV150', 'MP_POV150', 'EP_UNEMP', 'MP_UNEMP', 'EP_HBURD', 'MP_HBURD', 'EP_NOHSDP', 'MP_NOHSDP', 'EP_UNINSUR', 'MP_UNINSUR', 'EP_AGE65', 'MP_AGE65', 'EP_AGE17', 'MP_AGE17', 'EP_DISABL', 'MP_DISABL', 'EP_SNGPNT', 'MP_SNGPNT', 'EP_LIMENG', 'MP_LIMENG', 'EP_MINRTY', 'MP_MINRTY', 'EP_MUNIT', 'MP_MUNIT', 'EP_MOBILE', 'MP_MOBILE', 'EP_CROWD', 'MP_CROWD', 'EP_NOVEH', 'MP_NOVEH', 'EP_GROUPQ', 'MP_GROUPQ', 'EPL_POV150', 'EPL_UNEMP', 'EPL_HBURD', 'EPL_NOHSDP', 'EPL_UNINSUR', 'SPL_TH

In [20]:
# Keep only needed columns
SVI_COLS = [
    'ST_ABBR', 'STATE', 'E_TOTPOP',
    'RPL_THEMES', 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4',
    'EP_UNINSUR', 'EP_POV150', 'EP_UNEMP', 'EP_NOHSDP',
    'EP_AGE65', 'EP_DISABL', 'EP_MINRTY', 'EP_NOVEH', 'EP_NOINT'
]

svi = svi[SVI_COLS].copy()

# Replace -999 with NaN (SVI's null code)
svi = svi.replace(-999, np.nan)

print(f"Shape after column selection: {svi.shape}")
print(f"\nNull counts:")
print(svi.isnull().sum())

# Aggregate from county to state level (population-weighted average)
THEME_COLS = [
    'RPL_THEMES', 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4',
    'EP_UNINSUR', 'EP_POV150', 'EP_UNEMP', 'EP_NOHSDP',
    'EP_AGE65', 'EP_DISABL', 'EP_MINRTY', 'EP_NOVEH', 'EP_NOINT'
]

def weighted_avg(group):
    result = {}
    for col in THEME_COLS:
        valid = group[[col, 'E_TOTPOP']].dropna()
        if valid.empty or valid['E_TOTPOP'].sum() == 0:
            result[col] = np.nan
        else:
            result[col] = (valid[col] * valid['E_TOTPOP']).sum() / valid['E_TOTPOP'].sum()
    return pd.Series(result)

# ── FIX: include_groups=False prevents deprecation warning ───────────────────
svi_state = svi.groupby('ST_ABBR').apply(weighted_avg, include_groups=False).reset_index()

# Add state full name
state_names = svi.groupby('ST_ABBR')['STATE'].first().reset_index()
svi_state = svi_state.merge(state_names, on='ST_ABBR')

# Rename columns
svi_state = svi_state.rename(columns={
    'ST_ABBR':      'state_code',
    'STATE':        'state_name',
    'RPL_THEMES':   'svi_overall',
    'RPL_THEME1':   'svi_socioeconomic',
    'RPL_THEME2':   'svi_household',
    'RPL_THEME3':   'svi_minority',
    'RPL_THEME4':   'svi_housing_transport',
    'EP_UNINSUR':   'pct_uninsured',
    'EP_POV150':    'pct_poverty',
    'EP_UNEMP':     'pct_unemployed',
    'EP_NOHSDP':    'pct_no_hs_diploma',
    'EP_AGE65':     'pct_elderly',
    'EP_DISABL':    'pct_disabled',
    'EP_MINRTY':    'pct_minority',
    'EP_NOVEH':     'pct_no_vehicle',
    'EP_NOINT':     'pct_no_internet',
})

print(f"\nFinal SVI state-level shape: {svi_state.shape}")
print(f"\nNull counts:\n{svi_state.isnull().sum()}")
print(svi_state.head(10))

Shape after column selection: (3144, 17)

Null counts:
ST_ABBR       0
STATE         0
E_TOTPOP      0
RPL_THEMES    0
RPL_THEME1    0
RPL_THEME2    0
RPL_THEME3    0
RPL_THEME4    0
EP_UNINSUR    0
EP_POV150     0
EP_UNEMP      0
EP_NOHSDP     0
EP_AGE65      0
EP_DISABL     0
EP_MINRTY     0
EP_NOVEH      0
EP_NOINT      0
dtype: int64

Final SVI state-level shape: (51, 16)

Null counts:
state_code               0
svi_overall              0
svi_socioeconomic        0
svi_household            0
svi_minority             0
svi_housing_transport    0
pct_uninsured            0
pct_poverty              0
pct_unemployed           0
pct_no_hs_diploma        0
pct_elderly              0
pct_disabled             0
pct_minority             0
pct_no_vehicle           0
pct_no_internet          0
state_name               0
dtype: int64
  state_code  svi_overall  svi_socioeconomic  svi_household  svi_minority  \
0         AK     0.571324           0.451191       0.344832      0.761156   
1       

## Chosen Columns

| Column | Renamed To | What It Is |
|---|---|---|
| `ST_ABBR` | `state_code` | Join key to `dim_location` |
| `STATE` | `state_name` | State full name |
| `E_TOTPOP` | — | Population denominator for weighted averaging (dropped after aggregation) |
| `RPL_THEMES` | `svi_overall` | Overall composite vulnerability score (0–1) |
| `RPL_THEME1` | `svi_socioeconomic` | Socioeconomic vulnerability — poverty, unemployment, uninsured |
| `RPL_THEME2` | `svi_household` | Household composition — elderly, disabled, single parents |
| `RPL_THEME3` | `svi_minority` | Minority/language status |
| `RPL_THEME4` | `svi_housing_transport` | Housing & transportation barriers |
| `EP_UNINSUR` | `pct_uninsured` | % uninsured population |
| `EP_POV150` | `pct_poverty` | % below 150% poverty line |
| `EP_UNEMP` | `pct_unemployed` | % unemployed |
| `EP_NOHSDP` | `pct_no_hs_diploma` | % without high school diploma |
| `EP_AGE65` | `pct_elderly` | % aged 65+ |
| `EP_DISABL` | `pct_disabled` | % with a disability |
| `EP_MINRTY` | `pct_minority` | % minority population |
| `EP_NOVEH` | `pct_no_vehicle` | % with no vehicle |
| `EP_NOINT` | `pct_no_internet` | % with no internet access |

## Derived

| Derivation | What Was Done | Reasoning |
|---|---|---|
| `-999 → NaN` | Replaced SVI's null code with `NaN` before any calculation | Leaving -999 in would severely distort weighted averages |
| County → State aggregation | Population-weighted average of all county scores within each state using `E_TOTPOP` as weight | BRFSS only provides state geography — county-level SVI cannot join directly |

In [13]:
RENAME_MAP = {
    # Identifiers
    'SEQNO':            'respondent_id',
    '_STATE':           'state_fips',
    '_LLCPWT':          'survey_weight',

    # Interview date
    'IMONTH':           'interview_month',
    'IDAY':             'interview_day',
    'IYEAR':            'interview_year',

    # Geography
    '_METSTAT':         'metro_status',
    '_URBSTAT':         'urban_rural',
    'MSCODE':           'metro_division',

    # Demographics
    '_SEX':             'sex',
    '_AGEG5YR':         'age_group',
    '_RACE':            'race',
    '_HISPANC':         'hispanic',
    '_EDUCAG':          'education',
    '_INCOMG1':         'income_group',
    'MARITAL':          'marital_status',
    'EMPLOY1':          'employment',
    'CHILDREN':         'num_children',
    'VETERAN3':         'veteran',
    'PREGNANT':         'pregnant',

    # Health status
    'GENHLTH':          'general_health',
    'PHYSHLTH':         'phys_unhealthy_days',
    'MENTHLTH':         'mental_unhealthy_days',
    'LSATISFY':         'life_satisfaction',

    # BMI
    '_BMI5':            'bmi_raw',
    '_BMI5CAT':         'bmi_category',

    # Chronic conditions
    'CVDINFR4':         'heart_attack',
    'CVDSTRK3':         'stroke',
    'ASTHMA3':          'asthma',
    'ADDEPEV3':         'depression',
    'CHCCOPD3':         'copd',
    'HAVARTH4':         'arthritis',
    'CHCOCNC1':         'cancer',
    'CHCKDNY2':         'kidney_disease',

    # Behavioral risk factors
    'EXERANY2':         'exercise',
    '_SMOKER3':         'smoke_status',
    'ECIGNOW3':         'ecig_use',
    'DRNKANY6':         'any_alcohol',
    '_RFBING6':         'binge_drinking',
    '_RFDRHV9':         'heavy_drinker',
    'AVEDRNK4':         'avg_drinks_per_day',
    'MARIJAN1':         'marijuana_use',
    'SSBSUGR2':         'sugary_drinks',
    'FIREARM5':         'firearm_home',

    # Healthcare access
    '_HLTHPL2':         'has_insurance',
    'PRIMINS2':         'insurance_type',
    'PERSDOC3':         'has_pcp',
    'MEDCOST1':         'cost_barrier',
    'CHECKUP1':         'last_checkup',
    'LASTDEN4':         'last_dental',

    # Preventive care
    '_RFMAM23':         'mammogram_2yr',
    '_CRVSCRN':         'cervical_screen',
    '_CRCREC3':         'colorectal_screen',
    '_FLSHOT7':         'flu_shot',
    '_PNEUMO3':         'pneumo_vaccine',
    '_AIDTST4':         'hiv_test',

    # Social determinants
    'SDHFOOD1':         'food_insecure',
    'SDHTRNSP':         'transport_barrier',

    # Survey metadata
    'QSTVER':           'questionnaire_ver',
    'QSTLANG':          'language',
    'CELLFON5':         'cell_phone_only',
}

df = df.rename(columns=RENAME_MAP)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

Shape: (457670, 71)

Columns: ['respondent_id', 'state_fips', 'survey_weight', 'interview_month', 'interview_day', 'interview_year', 'metro_status', 'urban_rural', 'metro_division', 'sex', 'age_group', 'race', 'hispanic', 'education', 'income_group', 'marital_status', 'employment', 'num_children', 'veteran', 'pregnant', 'general_health', 'phys_unhealthy_days', 'mental_unhealthy_days', 'bmi_raw', 'bmi_category', 'heart_attack', 'stroke', 'asthma', 'depression', 'copd', 'arthritis', 'cancer', 'kidney_disease', 'life_satisfaction', 'exercise', 'smoke_status', 'ecig_use', 'any_alcohol', 'binge_drinking', 'heavy_drinker', 'avg_drinks_per_day', 'marijuana_use', 'sugary_drinks', 'firearm_home', 'has_insurance', 'insurance_type', 'has_pcp', 'cost_barrier', 'last_checkup', 'last_dental', 'mammogram_2yr', 'cervical_screen', 'colorectal_screen', 'flu_shot', 'pneumo_vaccine', 'hiv_test', 'food_insecure', 'transport_barrier', 'questionnaire_ver', 'language', 'cell_phone_only', 'diabetes_status', 'd

In [14]:
# FIPS → State lookup table
FIPS_LOOKUP = {
    1.0:  ('AL', 'Alabama'),
    2.0:  ('AK', 'Alaska'),
    4.0:  ('AZ', 'Arizona'),
    5.0:  ('AR', 'Arkansas'),
    6.0:  ('CA', 'California'),
    8.0:  ('CO', 'Colorado'),
    9.0:  ('CT', 'Connecticut'),
    10.0: ('DE', 'Delaware'),
    11.0: ('DC', 'District of Columbia'),
    12.0: ('FL', 'Florida'),
    13.0: ('GA', 'Georgia'),
    15.0: ('HI', 'Hawaii'),
    16.0: ('ID', 'Idaho'),
    17.0: ('IL', 'Illinois'),
    18.0: ('IN', 'Indiana'),
    19.0: ('IA', 'Iowa'),
    20.0: ('KS', 'Kansas'),
    21.0: ('KY', 'Kentucky'),
    22.0: ('LA', 'Louisiana'),
    23.0: ('ME', 'Maine'),
    24.0: ('MD', 'Maryland'),
    25.0: ('MA', 'Massachusetts'),
    26.0: ('MI', 'Michigan'),
    27.0: ('MN', 'Minnesota'),
    28.0: ('MS', 'Mississippi'),
    29.0: ('MO', 'Missouri'),
    30.0: ('MT', 'Montana'),
    31.0: ('NE', 'Nebraska'),
    32.0: ('NV', 'Nevada'),
    33.0: ('NH', 'New Hampshire'),
    34.0: ('NJ', 'New Jersey'),
    35.0: ('NM', 'New Mexico'),
    36.0: ('NY', 'New York'),
    37.0: ('NC', 'North Carolina'),
    38.0: ('ND', 'North Dakota'),
    39.0: ('OH', 'Ohio'),
    40.0: ('OK', 'Oklahoma'),
    41.0: ('OR', 'Oregon'),
    42.0: ('PA', 'Pennsylvania'),
    44.0: ('RI', 'Rhode Island'),
    45.0: ('SC', 'South Carolina'),
    46.0: ('SD', 'South Dakota'),
    47.0: ('TN', 'Tennessee'),
    48.0: ('TX', 'Texas'),
    49.0: ('UT', 'Utah'),
    50.0: ('VT', 'Vermont'),
    51.0: ('VA', 'Virginia'),
    53.0: ('WA', 'Washington'),
    54.0: ('WV', 'West Virginia'),
    55.0: ('WI', 'Wisconsin'),
    56.0: ('WY', 'Wyoming'),
    66.0: ('GU', 'Guam'),
    72.0: ('PR', 'Puerto Rico'),
    78.0: ('VI', 'Virgin Islands'),
}

# Add state_code and state_name to df
df['state_code'] = df['state_fips'].map(lambda x: FIPS_LOOKUP.get(x, (None, None))[0])
df['state_name'] = df['state_fips'].map(lambda x: FIPS_LOOKUP.get(x, (None, None))[1])

# Validate
print(f"Unique states: {df['state_code'].nunique()}")
print(f"Missing state_code: {df['state_code'].isna().sum()}")
print(f"\nSample:")
print(df[['state_fips', 'state_code', 'state_name']].drop_duplicates().sort_values('state_fips').head(10))

print(f"\nColumns: {df.columns.tolist()}")

Unique states: 53
Missing state_code: 0

Sample:
       state_fips state_code            state_name
0             1.0         AL               Alabama
5092          2.0         AK                Alaska
10588         4.0         AZ               Arizona
19176         5.0         AR              Arkansas
24520         6.0         CA            California
32902         8.0         CO              Colorado
43799         9.0         CT           Connecticut
51012        10.0         DE              Delaware
55372        11.0         DC  District of Columbia
58570        12.0         FL               Florida

Columns: ['respondent_id', 'state_fips', 'survey_weight', 'interview_month', 'interview_day', 'interview_year', 'metro_status', 'urban_rural', 'metro_division', 'sex', 'age_group', 'race', 'hispanic', 'education', 'income_group', 'marital_status', 'employment', 'num_children', 'veteran', 'pregnant', 'general_health', 'phys_unhealthy_days', 'mental_unhealthy_days', 'bmi_raw', 'bmi_catego

## FINAL COLUMNS

Based on everything we've done, here's the final updated list:

---

## BRFSS 2024 — 73 Columns

| Category | Columns |
|---|---|
| Identifier | `respondent_id` |
| Survey Metadata | `questionnaire_ver`, `language`, `cell_phone_only`, `interview_month`, `interview_day`, `interview_year` |
| Geography | `state_fips`, `state_code`, `state_name`, `metro_status`, `urban_rural`, `metro_division` |
| Demographics | `sex`, `age_group`, `race`, `hispanic`, `education`, `income_group`, `marital_status`, `employment`, `num_children`, `veteran`, `pregnant` |
| Health Status | `general_health`, `phys_unhealthy_days`, `mental_unhealthy_days`, `life_satisfaction` |
| BMI | `bmi_raw`, `bmi_category` |
| Chronic Conditions | `heart_attack`, `stroke`, `asthma`, `depression`, `copd`, `arthritis`, `cancer`, `kidney_disease`, `diabetes`, `diabetes_status` |
| Behavioral Risk Factors | `exercise`, `smoke_status`, `ecig_use`, `any_alcohol`, `binge_drinking`, `heavy_drinker`, `avg_drinks_per_day`, `marijuana_use`, `sugary_drinks`, `firearm_home` |
| Healthcare Access | `has_insurance`, `insurance_type`, `has_pcp`, `cost_barrier`, `last_checkup`, `last_dental` |
| Preventive Care | `mammogram_2yr`, `cervical_screen`, `colorectal_screen`, `flu_shot`, `pneumo_vaccine`, `hiv_test` |
| Social Determinants | `food_insecure`, `transport_barrier` |
| Survey Weight | `survey_weight` |
| Derived | `physhlth_bin`, `menthlth_bin`, `bmi_category_clean`, `ever_smoker`, `log_weight`, `condition_count`, `age_decade`, `quarter` |

---

## SVI 2022 — 16 Columns

| Category | Columns |
|---|---|
| Geography / Join Key | `state_code`, `state_name` |
| Overall Score | `svi_overall` |
| Theme Scores | `svi_socioeconomic`, `svi_household`, `svi_minority`, `svi_housing_transport` |
| Individual Indicators | `pct_uninsured`, `pct_poverty`, `pct_unemployed`, `pct_no_hs_diploma`, `pct_elderly`, `pct_disabled`, `pct_minority`, `pct_no_vehicle`, `pct_no_internet` |

---

## Medicaid Expansion — 3 Columns

| Category | Columns |
|---|---|
| Geography / Join Key | `state_name` |
| Policy Status | `medicaid_expansion_status` |
| Policy Detail | `expansion_year` |

In [30]:
import os

# ── CREATE OUTPUT FOLDER ──────────────────────────────────────
output_folder = folder + "/Transform Data"
os.makedirs(output_folder, exist_ok=True)

# ── SAVE BRFSS ────────────────────────────────────────────────
df.to_csv(output_folder + "/BRFSS_2024_transformed.csv", index=False)
print(f"✅ BRFSS saved     — {df.shape[0]:,} rows x {df.shape[1]} cols")

# ── SAVE SVI ──────────────────────────────────────────────────
svi_state.to_csv(output_folder + "/SVI_2022_state.csv", index=False)
print(f"✅ SVI saved       — {svi_state.shape[0]:,} rows x {svi_state.shape[1]} cols")

# ── SAVE MEDICAID ─────────────────────────────────────────────
medicaid.to_csv(output_folder + "/Medicaid_expansion_clean.csv", index=False)
print(f"✅ Medicaid saved  — {medicaid.shape[0]:,} rows x {medicaid.shape[1]} cols")

print(f"\nAll files saved to: {output_folder}")

✅ BRFSS saved     — 457,670 rows x 73 cols
✅ SVI saved       — 51 rows x 16 cols
✅ Medicaid saved  — 54 rows x 3 cols

All files saved to: /content/drive/MyDrive/MS Spring 2026 (Y1 Q3)/ADSP 31012 DEPA Final Project/Transform Data
